# Correlation Analysis: Normal vs ZCA-Whitened Embeddings

This notebook compares semantic trajectory metrics computed from:
- **Normal embeddings**: Raw embeddings from different models
- **ZCA-whitened embeddings**: Embeddings preprocessed with ZCA whitening

We analyze whether ZCA whitening changes the correlation structure and metric distributions.

In [ ]:
import sys
import os
import importlib
from pathlib import Path

from tabulate import tabulate
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

sys.path.append("../src")

import utils
import plot
import rstats

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)

## Configuration

In [ ]:
DATASETS = ["parkinson", "swear-fluency", "italian", "german"]
BACKENDS = ["openai", "gemini", "qwen", "fasttext-noncum"]
METRICS = ["d_next", "vel", "acc", "entropy", "d_centroid"]

# Readable names for plots
DATASET_NAMES = {
    "parkinson": "Neurodegenerative",
    "swear-fluency": "Swear Fluency",
    "italian": "Italian",
    "german": "German",
}

BACKEND_ABBREV = {
    "openai": "O",
    "gemini": "G",
    "qwen": "Q",
    "fasttext-noncum": "F",
}

METRIC_LATEX = {
    "d_next": r"$d_{\mathrm{next}}$",
    "vel": r"$\|\mathbf{v}\|$",
    "acc": r"$\|\mathbf{a}\|$",
    "entropy": r"$H$",
    "d_centroid": r"$d_{\mathrm{cent}}$",
}

RESULTS_DIR = Path("../results")
OUTPUT_DIR = RESULTS_DIR

## Load Data: Normal and ZCA

In [ ]:
def load_metrics(dataset, backend, use_zca=False):
    """Load metrics for a given dataset and backend.

    Args:
        dataset: Dataset name (e.g., 'parkinson', 'swear-fluency').
        backend: Embedding backend (e.g., 'openai', 'gemini').
        use_zca: If True, load from ZCA-whitened results.

    Returns:
        DataFrame with metrics, or None if file doesn't exist.
    """
    subdir = "zca" if use_zca else ""
    path = RESULTS_DIR / subdir / backend / dataset / "metrics.csv"

    if not path.exists():
        print(f"Warning: {path} not found")
        return None

    df = utils.load(str(path))[METRICS].reset_index(drop=True)
    return df


# Load all data
data_normal = {}
data_zca = {}

for dataset in DATASETS:
    data_normal[dataset] = {}
    data_zca[dataset] = {}

    for backend in BACKENDS:
        print(f"Loading {dataset} - {backend}")
        data_normal[dataset][backend] = load_metrics(dataset, backend, use_zca=False)
        data_zca[dataset][backend] = load_metrics(dataset, backend, use_zca=True)

## Boxplot Comparison: Normal vs ZCA

For each metric, we compare the distributions across all datasets and backends.

In [ ]:
def prepare_boxplot_data(metric):
    """Prepare data for boxplot comparison.

    Args:
        metric: Metric name to analyze.

    Returns:
        DataFrame with columns: value, preprocessing, dataset, backend.
    """
    rows = []

    for dataset in DATASETS:
        for backend in BACKENDS:
            # Normal data
            df_norm = data_normal[dataset][backend]
            if df_norm is not None and metric in df_norm.columns:
                values = df_norm[metric].dropna()
                for val in values:
                    rows.append(
                        {
                            "value": val,
                            "preprocessing": "Normal",
                            "dataset": DATASET_NAMES[dataset],
                            "backend": BACKEND_ABBREV[backend],
                        }
                    )

            # ZCA data
            df_zca = data_zca[dataset][backend]
            if df_zca is not None and metric in df_zca.columns:
                values = df_zca[metric].dropna()
                for val in values:
                    rows.append(
                        {
                            "value": val,
                            "preprocessing": "ZCA",
                            "dataset": DATASET_NAMES[dataset],
                            "backend": BACKEND_ABBREV[backend],
                        }
                    )

    return pd.DataFrame(rows)


# Create boxplots for all metrics
fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 6), sharey=False)

for ax, metric in zip(axes, METRICS):
    df_plot = prepare_boxplot_data(metric)

    sns.boxplot(
        data=df_plot,
        x="dataset",
        y="value",
        hue="preprocessing",
        ax=ax,
        palette="Set2",
    )

    ax.set_title(METRIC_LATEX[metric])
    ax.set_xlabel("")
    ax.set_ylabel("Value" if ax == axes[0] else "")
    ax.tick_params(axis="x", rotation=45)

    # Only show legend on first plot
    if ax != axes[0]:
        ax.get_legend().remove()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "boxplot-normal-vs-zca.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Correlation Heatmaps: Normal vs ZCA

We compute correlation matrices for each dataset, comparing normal and ZCA-whitened embeddings side by side.

In [ ]:
def compute_correlation_matrix(dataset, use_zca=False):
    """Compute correlation matrix across backends for a dataset.

    Args:
        dataset: Dataset name.
        use_zca: If True, use ZCA-whitened data.

    Returns:
        Correlation matrix DataFrame with renamed columns.
    """
    data_dict = data_zca if use_zca else data_normal

    # Load all backends for this dataset
    dfs = []
    for i, backend in enumerate(BACKENDS, start=1):
        df = data_dict[dataset][backend]
        if df is not None:
            df = df.add_suffix(f"_m{i}")
            dfs.append(df)

    # Combine and align
    combined = dfs[0]
    for df in dfs[1:]:
        combined = combined.join(df, how="inner")

    # Reorder columns by metric type
    ordered_cols = []
    for metric in METRICS:
        for i in range(1, len(BACKENDS) + 1):
            col = f"{metric}_m{i}"
            if col in combined.columns:
                ordered_cols.append(col)

    combined = combined[ordered_cols]

    # Rename columns with LaTeX formatting
    mapping = {}
    for metric in METRICS:
        for i, backend in enumerate(BACKENDS, start=1):
            old_name = f"{metric}_m{i}"
            new_name = f"{METRIC_LATEX[metric]} ({BACKEND_ABBREV[backend]})"
            mapping[old_name] = new_name

    combined = combined.rename(columns=mapping)

    return combined.corr()


# Compute correlations
corrs_normal = [compute_correlation_matrix(ds, use_zca=False) for ds in DATASETS]
corrs_zca = [compute_correlation_matrix(ds, use_zca=True) for ds in DATASETS]

### Heatmap Visualization: Side-by-Side Comparison

In [ ]:
# Create 2-row layout: Normal (top) vs ZCA (bottom)
fig, axes = plt.subplots(
    2, len(DATASETS), figsize=(5 * len(DATASETS), 10), sharex=False, sharey=False
)

vmin, vmax = -1, 1
cmap = "coolwarm"

# Plot normal correlations (top row)
for col, (corr, dataset) in enumerate(zip(corrs_normal, DATASETS)):
    ax = axes[0, col]
    hm = sns.heatmap(
        corr,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        annot=False,
        square=True,
        cbar=False,
        ax=ax,
    )

    ax.set_title(f"{DATASET_NAMES[dataset]} (Normal)")
    ax.set_xticks(np.arange(len(corr.columns)) + 0.5)
    ax.set_xticklabels(corr.columns, rotation=90, ha="center", fontsize=9)

    if col == 0:
        ax.set_yticklabels(corr.index, rotation=0, fontsize=9)
    else:
        ax.set_yticks([])
        ax.set_ylabel("")

# Plot ZCA correlations (bottom row)
for col, (corr, dataset) in enumerate(zip(corrs_zca, DATASETS)):
    ax = axes[1, col]
    hm = sns.heatmap(
        corr,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        annot=False,
        square=True,
        cbar=False,
        ax=ax,
    )

    ax.set_title(f"{DATASET_NAMES[dataset]} (ZCA)")
    ax.set_xticks(np.arange(len(corr.columns)) + 0.5)
    ax.set_xticklabels(corr.columns, rotation=90, ha="center", fontsize=9)

    if col == 0:
        ax.set_yticklabels(corr.index, rotation=0, fontsize=9)
    else:
        ax.set_yticks([])
        ax.set_ylabel("")

# Add common colorbar
cax = fig.add_axes([0.25, -0.02, 0.5, 0.015])
fig.colorbar(
    hm.collections[0],
    cax=cax,
    orientation="horizontal",
)

plt.subplots_adjust(wspace=0.05, hspace=0.25)
plt.savefig(
    OUTPUT_DIR / "correlation-heatmaps-comparison.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

## Correlation Difference Analysis

We compute the difference between normal and ZCA correlation matrices to quantify the impact of whitening.

In [ ]:
# Compute differences
corr_diffs = []
for corr_norm, corr_zca in zip(corrs_normal, corrs_zca):
    diff = corr_zca - corr_norm
    corr_diffs.append(diff)

# Visualize differences
fig, axes = plt.subplots(
    1, len(DATASETS), figsize=(5 * len(DATASETS), 5), sharex=False, sharey=False
)

vmin_diff, vmax_diff = -0.5, 0.5
cmap_diff = "RdBu_r"

for ax, diff, dataset in zip(axes, corr_diffs, DATASETS):
    hm = sns.heatmap(
        diff,
        cmap=cmap_diff,
        vmin=vmin_diff,
        vmax=vmax_diff,
        annot=False,
        square=True,
        cbar=False,
        ax=ax,
        center=0,
    )

    ax.set_title(f"{DATASET_NAMES[dataset]}")
    ax.set_xticks(np.arange(len(diff.columns)) + 0.5)
    ax.set_xticklabels(diff.columns, rotation=90, ha="center")

    if ax == axes[0]:
        ax.set_yticklabels(diff.index, rotation=0)
    else:
        ax.set_yticks([])
        ax.set_ylabel("")

# Add colorbar
cax = fig.add_axes([0.25, -0.1, 0.5, 0.03])
fig.colorbar(
    hm.collections[0],
    cax=cax,
    orientation="horizontal",
)

plt.subplots_adjust(wspace=0.05)
plt.savefig(OUTPUT_DIR / "correlation-differences.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Summary Statistics

In [ ]:
# Compute summary statistics for each metric
summary_rows = []

for metric in METRICS:
    df_plot = prepare_boxplot_data(metric)

    # Aggregate across all datasets and backends
    normal_vals = df_plot[df_plot["preprocessing"] == "Normal"]["value"]
    zca_vals = df_plot[df_plot["preprocessing"] == "ZCA"]["value"]

    summary_rows.append(
        {
            "Metric": metric,
            "Normal Mean": normal_vals.mean(),
            "Normal Std": normal_vals.std(),
            "ZCA Mean": zca_vals.mean(),
            "ZCA Std": zca_vals.std(),
            "Mean Difference": zca_vals.mean() - normal_vals.mean(),
            "Std Difference": zca_vals.std() - normal_vals.std(),
        }
    )

summary_df = pd.DataFrame(summary_rows)
print("\nSummary Statistics: Normal vs ZCA")
print(tabulate(summary_df, headers="keys", tablefmt="github", floatfmt=".4f"))

## Correlation Change Summary

Quantify how much correlations changed due to ZCA whitening.

In [ ]:
change_summary = []

for dataset, diff in zip(DATASETS, corr_diffs):
    # Extract upper triangle (exclude diagonal)
    mask = np.triu(np.ones_like(diff, dtype=bool), k=1)
    upper_tri = diff.values[mask]

    change_summary.append(
        {
            "Dataset": DATASET_NAMES[dataset],
            "Mean Abs Change": np.abs(upper_tri).mean(),
            "Max Abs Change": np.abs(upper_tri).max(),
            "% Changed > 0.1": (np.abs(upper_tri) > 0.1).mean() * 100,
            "% Changed > 0.2": (np.abs(upper_tri) > 0.2).mean() * 100,
        }
    )

change_df = pd.DataFrame(change_summary)
print("\nCorrelation Structure Changes (ZCA - Normal)")
print(tabulate(change_df, headers="keys", tablefmt="github", floatfmt=".4f"))